### Generate images

In [ ]:
# !pip install -q huggingface_hub
# !pip install -q datasets

In [ ]:
from huggingface_hub import login
from datasets import load_dataset
from datasets import Features, Value, ClassLabel, Image


login(token="")#Your token here

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

model_id = "stabilityai/stable-diffusion-2-1"

# Use the DPMSolverMultistepScheduler (DPM-Solver++) scheduler here instead
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

In [ ]:
pipe = pipe.to("cuda")

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
prompts_data_frame = pd.read_csv("datasets/prompts.csv", sep=";")

In [ ]:
def generate_images(row, pipe, number_of_images):
  for i in range(number_of_images):
    dir_name = f"{row.array[0]}/{row.array[1]}/{row.array[2]}"
    dirpath = Path(dir_name)
    neutral_path = dirpath / "neutral_image"
    neutral_path.mkdir(parents=True, exist_ok=True)
    bias_aware_path = dirpath / "bias_aware_prompt"
    bias_aware_path.mkdir(parents=True, exist_ok=True)
    llm_prompt_path = dirpath / "llm_prompt"
    llm_prompt_path.mkdir(parents=True, exist_ok=True)
    neutral_image = pipe(row.array[2]).images[0]
    neutral_image.save(neutral_path / f"{i}.jpg")
    bias_aware_prompt_image = pipe(row.array[3]).images[0]
    bias_aware_prompt_image.save(bias_aware_path/ f"{i}.jpg")
    llm_prompt_image = pipe(row.array[4]).images[0]
    llm_prompt_image.save(llm_prompt_path / f"{i}.jpg")

In [ ]:
def generate_images_from_df(pipe, prompt_data_frame, number_of_images):
  prompt_data_frame.apply(generate_images, axis=1 ,args=(pipe, number_of_images))

In [ ]:
generate_images_from_df(pipe, prompts_data_frame, 5)

### Create a HuggingFace Dataset from the generated images

In [1]:
import pandas as pd
from pathlib import Path
from PIL import Image
from datasets import Dataset, Features, Value, ClassLabel, Image as HFImage


/opt/miniconda3/envs/eeai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
root_dir = Path(".")
data = []

In [ ]:
for main_category in root_dir.iterdir():
    if main_category.is_dir():
        for sub_category in main_category.iterdir():
            if sub_category.is_dir():
                for prompt in sub_category.iterdir():
                    if prompt.is_dir():
                        for strategy in ["neutral_image", "bias_aware_prompt", "llm_prompt"]:
                            strategy_dir = prompt / strategy
                            if strategy_dir.exists():
                                for img_file in strategy_dir.glob("*.jpg"):
                                    try:
                                        # Try to open and load the image (less strict than verify)
                                        with Image.open(img_file) as img:
                                            img.load()
                                        data.append({
                                            "Image": str(img_file),
                                            "Prompt": prompt.name,
                                            "Main Category": main_category.name,
                                            "Sub-Category": sub_category.name,
                                            "Prompt Strategies": strategy
                                        })
                                    except Exception as e:
                                        print(f"Skipping invalid image: {img_file} ({e})")
df = pd.DataFrame(data)

                                               Image  \
0  Geo-Cultural Bias/temp/A traditional dance per...   
1  Geo-Cultural Bias/temp/A traditional dance per...   
2  Geo-Cultural Bias/temp/A traditional dance per...   
3  Geo-Cultural Bias/temp/A traditional dance per...   
4  Geo-Cultural Bias/temp/A traditional dance per...   

                             Prompt      Main Category Sub-Category  \
0  A traditional dance performance.  Geo-Cultural Bias         temp   
1  A traditional dance performance.  Geo-Cultural Bias         temp   
2  A traditional dance performance.  Geo-Cultural Bias         temp   
3  A traditional dance performance.  Geo-Cultural Bias         temp   
4  A traditional dance performance.  Geo-Cultural Bias         temp   

  Prompt Strategies  
0     neutral_image  
1     neutral_image  
2     neutral_image  
3     neutral_image  
4     neutral_image  


In [4]:
features = Features({
    "Image": HFImage(),
    "Prompt": Value("string"),
    "Main Category": Value("string"),
    "Sub-Category": Value("string"),
    "Prompt Strategies": ClassLabel(names=["neutral_image", "bias_aware_prompt", "llm_prompt"])
})

In [5]:
hf_dataset = Dataset.from_pandas(df, features=features)

hf_dataset.push_to_hub("Sleepwalker467/stable_diffusion_generated")

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/Sleepwalker467/stable_diffusion_generated/commit/59a47aee1c93f6fe5440173ff1012c484ed71839', commit_message='Upload dataset', commit_description='', oid='59a47aee1c93f6fe5440173ff1012c484ed71839', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Sleepwalker467/stable_diffusion_generated', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Sleepwalker467/stable_diffusion_generated'), pr_revision=None, pr_num=None)